In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, count, avg, desc, round as spark_round, collect_list, concat, lit, size
from pyspark.sql.window import Window

# Khởi tạo Spark Session
spark = (SparkSession.builder
    .appName("TFT-Traits-Tier-List")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

# Đọc dữ liệu
print(">>> Đang đọc dữ liệu...")
df = spark.read.option("recursiveFileLookup", "true") \
               .option("inferSchema", "true") \
               .json("./data_3580_matches")

# Xử lý lớp bọc 'data' nếu có
if "data" in df.columns and "info" not in df.columns:
    df = df.select("data.*")

df.cache()
print(f"Done! Đã load {df.count()} trận đấu.")

your 131072x1 screen size is bogus. expect trouble
25/12/02 12:56:23 WARN Utils: Your hostname, DESKTOP-O957G4G resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/02 12:56:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/12/02 12:56:23 WARN Utils: Your hostname, DESKTOP-O957G4G resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/02 12:56:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/02 12:56:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/02 12:56:26 WARN NativeCodeLoader: Unable to load native-hadoo

>>> Đang đọc dữ liệu...


25/12/02 12:56:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Done! Đã load 3580 trận đấu.


## Kiểm tra cấu trúc Traits trong dữ liệu

In [2]:
from pyspark.sql.functions import col, explode

# Kiểm tra cấu trúc dữ liệu traits
print("=== KIỂM TRA CẤU TRÚC DỮ LIỆU TRAITS ===\n")

# Lấy mẫu 1 người chơi để xem cấu trúc traits
sample = df.select(
    explode(col("info.participants")).alias("player")
).select(
    col("player.placement"),
    col("player.traits")
).limit(1)

sample.show(1, truncate=80, vertical=True)

# Xem chi tiết cấu trúc 1 trait
print("\n=== MẪU CHI TIẾT 1 TRAIT ===\n")
sample_trait = df.select(
    explode(col("info.participants")).alias("player")
).select(
    col("player.placement"),
    explode(col("player.traits")).alias("trait")
).select(
    col("placement"),
    col("trait.name").alias("trait_name"),
    col("trait.num_units").alias("num_units"),
    col("trait.tier_current").alias("tier_current"),
    col("trait.tier_total").alias("tier_total"),
    col("trait.style").alias("style")
).limit(10)

sample_trait.show(10, truncate=False)

=== KIỂM TRA CẤU TRÚC DỮ LIỆU TRAITS ===



-RECORD 0-------------------------------------------------------------------------------------
 placement | 8                                                                                
 traits    | [{TFT15_Destroyer, 3, 2, 2, 4}, {TFT15_DragonFist, 1, 3, 1, 1}, {TFT15_Duelis... 


=== MẪU CHI TIẾT 1 TRAIT ===

+---------+------------------+---------+------------+----------+-----+
|placement|trait_name        |num_units|tier_current|tier_total|style|
+---------+------------------+---------+------------+----------+-----+
|8        |TFT15_Destroyer   |3        |2           |4         |2    |
|8        |TFT15_DragonFist  |1        |1           |1         |3    |
|8        |TFT15_Duelist     |1        |0           |3         |0    |
|8        |TFT15_Heavyweight |1        |0           |3         |0    |
|8        |TFT15_Juggernaut  |1        |0           |3         |0    |
|8        |TFT15_OldMentor   |1        |1           |3         |1    |
|8        |TFT15_SentaiRanger|6        |2   

## Bước 1: Phẳng hóa dữ liệu Traits (Flatten Trait Data)

Chuyển đổi dữ liệu từ cấu trúc lồng nhau thành dạng phẳng để phân tích. Mỗi dòng sẽ đại diện cho 1 trait được kích hoạt trong game.

**Lưu ý quan trọng:**
- `tier_current`: Mức kích hoạt hiện tại (0 = chưa kích hoạt)
- `tier_total`: Số mức tối đa của trait
- `style`: Style hiển thị (0=none, 1=bronze, 2=silver, 3=gold, 4=chromatic)
- Chỉ tính những trait có `tier_current > 0` (đã kích hoạt)

In [3]:
# Bước 1: Bung mảng participants (8 người chơi)
df_players = df.select(
    explode(col("info.participants")).alias("player")
).select(
    col("player.placement"),
    col("player.traits")
)

# Bước 2: Bung mảng traits của mỗi người chơi
df_traits = df_players.select(
    col("placement"),
    explode(col("traits")).alias("trait")
).select(
    col("placement"),
    col("trait.name").alias("trait_id"),
    col("trait.num_units").alias("num_units"),
    col("trait.tier_current").alias("tier_current"),
    col("trait.tier_total").alias("tier_total"),
    col("trait.style").alias("style")
).filter(
    # Chỉ lấy những trait ĐÃ KÍCH HOẠT (tier_current > 0)
    col("tier_current") > 0
)

# Tạo cột win (Top 4 = Win)
df_traits = df_traits.withColumn(
    "win", 
    (col("placement") <= 4).cast("int")
)

print(f"Tổng số traits đã kích hoạt tìm thấy: {df_traits.count():,}")
df_traits.show(10)

Tổng số traits đã kích hoạt tìm thấy: 169,164
+---------+------------------+---------+------------+----------+-----+---+
|placement|          trait_id|num_units|tier_current|tier_total|style|win|
+---------+------------------+---------+------------+----------+-----+---+
|        8|   TFT15_Destroyer|        3|           2|         4|    2|  0|
|        8|  TFT15_DragonFist|        1|           1|         1|    3|  0|
|        8|   TFT15_OldMentor|        1|           1|         3|    1|  0|
|        8|TFT15_SentaiRanger|        6|           2|         4|    2|  0|
|        8|TFT15_Spellslinger|        2|           1|         3|    1|  0|
|        8|  TFT15_Strategist|        2|           1|         4|    1|  0|
|        7|    TFT15_GemForce|        7|           3|         4|    4|  0|
|        7|   TFT15_Protector|        2|           1|         3|    1|  0|
|        7|  TFT15_Rosemother|        1|           1|         1|    3|  0|
|        7|TFT15_Spellslinger|        2|           1| 

## Bước 2: Tính toán chỉ số thống kê cho Traits

Tính toán các chỉ số quan trọng:
- **Count**: Số lần trait được kích hoạt
- **Avg Place**: Thứ hạng trung bình (thấp = tốt)
- **Win Rate**: Tỷ lệ Top 4 (%)
- **Frequency**: Tần suất xuất hiện (%)

In [4]:
# Tổng số traits để tính frequency
total_traits = df_traits.count()

# Thống kê cơ bản cho mỗi trait
trait_stats = df_traits.groupBy("trait_id").agg(
    count("*").alias("count"),
    spark_round(avg("placement"), 2).alias("avg_place"),
    spark_round(avg("win") * 100, 1).alias("win_rate"),
    spark_round(avg("tier_current"), 1).alias("avg_tier"),
    spark_round(avg("num_units"), 1).alias("avg_units")
).withColumn(
    "frequency_pct", 
    spark_round((col("count") / total_traits) * 100, 2)
)

print("=== TOP 10 TRAITS (Sắp xếp theo Avg Place) ===")
trait_stats.orderBy("avg_place").show(10, truncate=False)

=== TOP 10 TRAITS (Sắp xếp theo Avg Place) ===


+----------------+-----+---------+--------+--------+---------+-------------+
|trait_id        |count|avg_place|win_rate|avg_tier|avg_units|frequency_pct|
+----------------+-----+---------+--------+--------+---------+-------------+
|TFT15_DragonFist|4822 |3.66     |65.0    |1.0     |1.0      |2.85         |
|TFT15_ElTigre   |7667 |3.76     |63.2    |1.0     |1.0      |4.53         |
|TFT15_Luchador  |4481 |3.82     |62.9    |1.1     |2.2      |2.65         |
|TFT15_Captain   |3805 |3.88     |59.7    |1.0     |1.0      |2.25         |
|TFT15_Rosemother|6503 |3.92     |59.0    |1.0     |1.0      |3.84         |
|TFT15_Destroyer |6865 |4.1      |58.8    |1.5     |2.5      |4.06         |
|TFT15_Bastion   |11368|4.19     |55.4    |1.5     |3.0      |6.72         |
|TFT15_Sniper    |3349 |4.2      |56.2    |1.3     |2.3      |1.98         |
|TFT15_Edgelord  |4322 |4.2      |57.3    |1.1     |2.3      |2.55         |
|TFT15_TheCrew   |2119 |4.21     |54.4    |7.0     |3.1      |1.25         |

## Bước 3: Phân tích theo Tier Level của Trait

Xem hiệu quả của mỗi trait ở từng mức kích hoạt khác nhau (Tier 1, 2, 3, 4...).

In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, concat_ws

# Thống kê theo tier_current cho mỗi trait
trait_tier_stats = df_traits.groupBy("trait_id", "tier_current").agg(
    count("*").alias("count"),
    spark_round(avg("placement"), 2).alias("avg_place"),
    spark_round(avg("win") * 100, 1).alias("win_rate")
)

# Tạo window để rank theo từng trait
window_spec = Window.partitionBy("trait_id").orderBy(desc("count"))

# Tổng hợp các tier levels phổ biến cho mỗi trait
popular_tiers = trait_tier_stats.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank") <= 3  # Top 3 tier levels phổ biến nhất
).groupBy("trait_id").agg(
    collect_list(
        concat(lit("T"), col("tier_current").cast("string"))
    ).alias("popular_tiers")
)

print("=== POPULAR TIER LEVELS CHO MỖI TRAIT ===")
popular_tiers.show(10, truncate=False)

=== POPULAR TIER LEVELS CHO MỖI TRAIT ===
+--------------------+-------------+
|trait_id            |popular_tiers|
+--------------------+-------------+
|TFT15_Bastion       |[T1, T3, T2] |
|TFT15_BattleAcademia|[T3, T2, T1] |
|TFT15_Captain       |[T1]         |
|TFT15_Destroyer     |[T1, T2, T3] |
|TFT15_DragonFist    |[T1]         |
|TFT15_Duelist       |[T1, T3, T2] |
|TFT15_Edgelord      |[T1, T2, T3] |
|TFT15_ElTigre       |[T1]         |
|TFT15_Empyrean      |[T1, T3, T2] |
|TFT15_GemForce      |[T2, T1, T3] |
+--------------------+-------------+
only showing top 10 rows

+--------------------+-------------+
|trait_id            |popular_tiers|
+--------------------+-------------+
|TFT15_Bastion       |[T1, T3, T2] |
|TFT15_BattleAcademia|[T3, T2, T1] |
|TFT15_Captain       |[T1]         |
|TFT15_Destroyer     |[T1, T2, T3] |
|TFT15_DragonFist    |[T1]         |
|TFT15_Duelist       |[T1, T3, T2] |
|TFT15_Edgelord      |[T1, T2, T3] |
|TFT15_ElTigre       |[T1]         |
|TFT15_

## Bước 4: Tính Baseline Placement và Place Change

Tính placement trung bình của người chơi KHÔNG có trait cụ thể, sau đó so sánh với khi CÓ trait đó.

**Place Change** = Avg Place (with trait) - Baseline Place (without trait)
- **Âm** = Trait giúp placement tốt hơn
- **Dương** = Trait không hiệu quả

In [6]:
from pyspark.sql.functions import sum as spark_sum, collect_set, array_contains

# Tính placement trung bình tổng thể (baseline overall)
overall_avg_placement = df.select(
    explode(col("info.participants")).alias("player")
).select(
    col("player.placement")
).agg(
    spark_round(avg("placement"), 2).alias("overall_avg")
).first()["overall_avg"]

print(f"Placement trung bình tổng thể: {overall_avg_placement}")

# Lấy danh sách tất cả traits active của mỗi player
df_player_traits = df.select(
    explode(col("info.participants")).alias("player")
).select(
    col("player.placement"),
    col("player.traits")
)

# Bung ra và lấy danh sách traits active
df_player_active_traits = df_player_traits.select(
    col("placement"),
    explode(col("traits")).alias("trait")
).filter(
    col("trait.tier_current") > 0
).groupBy("placement").agg(
    collect_set("trait.name").alias("active_traits")
)

print("\n=== MẪU PLAYER VỚI DANH SÁCH TRAITS ACTIVE ===")
df_player_active_traits.show(5, truncate=60)

Placement trung bình tổng thể: 4.47

=== MẪU PLAYER VỚI DANH SÁCH TRAITS ACTIVE ===
+---------+------------------------------------------------------------+
|placement|                                               active_traits|
+---------+------------------------------------------------------------+
|        7|[TFT15_Edgelord, TFT15_TheCrew, TFT15_Strategist, TFT15_D...|
|        6|[TFT15_Edgelord, TFT15_TheCrew, TFT15_Strategist, TFT15_D...|
|        5|[TFT15_Edgelord, TFT15_TheCrew, TFT15_Strategist, TFT15_D...|
|        1|[TFT15_Edgelord, TFT15_TheCrew, TFT15_Strategist, TFT15_D...|
|        3|[TFT15_Edgelord, TFT15_TheCrew, TFT15_Strategist, TFT15_D...|
+---------+------------------------------------------------------------+
only showing top 5 rows

+---------+------------------------------------------------------------+
|placement|                                               active_traits|
+---------+------------------------------------------------------------+
|        7|[TFT

In [7]:
# Lấy tất cả placements từ players
all_player_placements = df.select(
    explode(col("info.participants")).alias("player")
).select(
    col("player.placement")
)

# Tính baseline: Placement TB của player KHÔNG có trait đó active
# Approach: Với mỗi trait, tính avg placement của players KHÔNG có trait đó

# Lấy tất cả unique traits
all_traits = df_traits.select("trait_id").distinct().collect()
trait_list = [row["trait_id"] for row in all_traits]

print(f"Tổng số traits: {len(trait_list)}")
print(f"\nCác traits: {trait_list[:10]}...")

Tổng số traits: 26

Các traits: ['TFT15_Bastion', 'TFT15_Strategist', 'TFT15_DragonFist', 'TFT15_StarGuardian', 'TFT15_Spellslinger', 'TFT15_GemForce', 'TFT15_Edgelord', 'TFT15_Sniper', 'TFT15_Empyrean', 'TFT15_ElTigre']...


In [8]:
# Tính baseline cho mỗi trait (avg placement khi KHÔNG có trait)
# Sử dụng cách tiếp cận: So sánh với overall average

# Tính weighted Place Change dựa trên tier level
# Mỗi tier level có thể có hiệu quả khác nhau

trait_place_change = df_traits.groupBy("trait_id").agg(
    spark_round(avg("placement"), 2).alias("avg_place_with_trait"),
    count("*").alias("count")
).withColumn(
    "place_change",
    spark_round(col("avg_place_with_trait") - lit(overall_avg_placement), 2)
)

print("=== PLACE CHANGE CỦA MỖI TRAIT ===")
print(f"(So với baseline trung bình: {overall_avg_placement})")
print("(Giá trị ÂM = Trait giúp placement tốt hơn)\n")
trait_place_change.orderBy("place_change").show(20, truncate=False)

=== PLACE CHANGE CỦA MỖI TRAIT ===
(So với baseline trung bình: 4.47)
(Giá trị ÂM = Trait giúp placement tốt hơn)



+--------------------+--------------------+-----+------------+
|trait_id            |avg_place_with_trait|count|place_change|
+--------------------+--------------------+-----+------------+
|TFT15_DragonFist    |3.66                |4822 |-0.81       |
|TFT15_ElTigre       |3.76                |7667 |-0.71       |
|TFT15_Luchador      |3.82                |4481 |-0.65       |
|TFT15_Captain       |3.88                |3805 |-0.59       |
|TFT15_Rosemother    |3.92                |6503 |-0.55       |
|TFT15_Destroyer     |4.1                 |6865 |-0.37       |
|TFT15_Bastion       |4.19                |11368|-0.28       |
|TFT15_Edgelord      |4.2                 |4322 |-0.27       |
|TFT15_Sniper        |4.2                 |3349 |-0.27       |
|TFT15_TheCrew       |4.21                |2119 |-0.26       |
|TFT15_SoulFighter   |4.22                |5402 |-0.25       |
|TFT15_SentaiRanger  |4.23                |5931 |-0.24       |
|TFT15_Juggernaut    |4.23                |8910 |-0.24 

## Bước 5: Kết hợp dữ liệu và tạo Tier List

Join các bảng thống kê lại với nhau và tạo bảng xếp hạng Traits theo Place Change.

In [9]:
# Join trait stats với popular tiers và place_change
final_result = trait_stats.join(popular_tiers, "trait_id", "left") \
                          .join(trait_place_change.select("trait_id", "place_change"), "trait_id", "left") \
                          .orderBy("place_change")

# Tạo cột Frequency hiển thị
display_df = final_result.select(
    col("trait_id").alias("Trait"),
    col("avg_place").alias("Avg Place"),
    col("place_change").alias("Place Change"),
    col("win_rate").alias("Win Rate %"),
    col("avg_tier").alias("Avg Tier"),
    concat(col("count"), lit(" ("), col("frequency_pct"), lit("%)")).alias("Frequency"),
    col("popular_tiers").alias("Popular Tiers")
)

print("=" * 80)
print("TFT TRAITS TIER LIST - SET 15")
print("=" * 80)
print(f"Tổng số traits phân tích: {display_df.count()}")
print(f"Tổng số lần traits được kích hoạt: {total_traits:,}")
print("=" * 80)
print("\nTOP 20 TRAITS TỐT NHẤT (Theo Place Change):\n")
display_df.show(20, truncate=False)

TFT TRAITS TIER LIST - SET 15
Tổng số traits phân tích: 26
Tổng số lần traits được kích hoạt: 169,164

TOP 20 TRAITS TỐT NHẤT (Theo Place Change):

Tổng số traits phân tích: 26
Tổng số lần traits được kích hoạt: 169,164

TOP 20 TRAITS TỐT NHẤT (Theo Place Change):



+--------------------+---------+------------+----------+--------+--------------+-------------+
|Trait               |Avg Place|Place Change|Win Rate %|Avg Tier|Frequency     |Popular Tiers|
+--------------------+---------+------------+----------+--------+--------------+-------------+
|TFT15_DragonFist    |3.66     |-0.81       |65.0      |1.0     |4822 (2.85%)  |[T1]         |
|TFT15_ElTigre       |3.76     |-0.71       |63.2      |1.0     |7667 (4.53%)  |[T1]         |
|TFT15_Luchador      |3.82     |-0.65       |62.9      |1.1     |4481 (2.65%)  |[T1, T2]     |
|TFT15_Captain       |3.88     |-0.59       |59.7      |1.0     |3805 (2.25%)  |[T1]         |
|TFT15_Rosemother    |3.92     |-0.55       |59.0      |1.0     |6503 (3.84%)  |[T1]         |
|TFT15_Destroyer     |4.1      |-0.37       |58.8      |1.5     |6865 (4.06%)  |[T1, T2, T3] |
|TFT15_Bastion       |4.19     |-0.28       |55.4      |1.5     |11368 (6.72%) |[T1, T3, T2] |
|TFT15_Edgelord      |4.2      |-0.27       |57.3 

## Bước 6: Lọc Traits phổ biến và tạo UI Table

In [10]:
import pandas as pd
from IPython.display import display, HTML

# Lọc traits có ít nhất 100 lần xuất hiện
min_count = 100

popular_traits = final_result.filter(col("count") >= min_count)

# Sắp xếp theo Place Change
popular_traits_sorted = popular_traits.orderBy("place_change")

# Chuyển sang Pandas để tạo bảng UI đẹp
pdf = popular_traits_sorted.limit(50).toPandas()

# Làm ngắn tên trait
pdf['Trait'] = pdf['trait_id'].str.replace('TFT15_', '').str.replace('tft15_', '')

# Lấy popular tiers
pdf['Levels'] = pdf['popular_tiers'].apply(lambda x: ', '.join(x[:3]) if x else 'N/A')

# Tạo DataFrame hiển thị
display_pdf = pdf[['Trait', 'avg_place', 'place_change', 'win_rate', 'avg_tier', 'count', 'frequency_pct', 'Levels']].copy()
display_pdf.columns = ['Trait', 'Avg Place', 'Place Change', 'Win Rate %', 'Avg Tier', 'Count', 'Freq %', 'Levels']

# Tạo HTML table với styling
def create_styled_table(df):
    html = """
    <style>
        .tft-table {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            border-collapse: collapse;
            width: 100%;
            margin: 20px 0;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }
        .tft-table th {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 12px 15px;
            text-align: left;
            font-weight: 600;
        }
        .tft-table td {
            padding: 10px 15px;
            border-bottom: 1px solid #ddd;
        }
        .tft-table tr:nth-child(even) {
            background-color: #f8f9fa;
        }
        .tft-table tr:hover {
            background-color: #e9ecef;
        }
        .positive { color: #ef4444; font-weight: bold; }
        .negative { color: #22c55e; font-weight: bold; }
        .tier-s { background-color: #ffd700; color: #000; font-weight: bold; }
        .tier-a { background-color: #c0c0c0; color: #000; font-weight: bold; }
    </style>
    <table class="tft-table">
    <thead><tr>
    """
    
    # Header
    html += "<th>#</th>"
    for col in df.columns:
        html += f"<th>{col}</th>"
    html += "</tr></thead><tbody>"
    
    # Rows
    for idx, row in df.iterrows():
        html += f"<tr><td><b>{idx + 1}</b></td>"
        for col in df.columns:
            value = row[col]
            if col == 'Place Change':
                css_class = 'negative' if value < 0 else 'positive'
                html += f"<td class='{css_class}'>{value:+.2f}</td>"
            elif col == 'Win Rate %':
                html += f"<td><b>{value:.1f}%</b></td>"
            elif col == 'Avg Place':
                html += f"<td><b>{value:.2f}</b></td>"
            else:
                html += f"<td>{value}</td>"
        html += "</tr>"
    
    html += "</tbody></table>"
    return html

print("=" * 80)
print("📊 TFT TRAITS TIER LIST - SET 15 (Sorted by Trait Impact)")
print("=" * 80)
print(f"📈 Tổng số traits phân tích: {popular_traits_sorted.count()}")
print(f"📊 Hiển thị Top 50 traits có impact tốt nhất")
print("=" * 80)
print("\n🎨 Chú thích Place Change:")
print("  🟢 Place Change ÂM (xanh) = Trait giúp placement TỐT HƠN")
print("  🔴 Place Change DƯƠNG (đỏ) = Trait làm placement KÉM HƠN")
print("=" * 80 + "\n")

display(HTML(create_styled_table(display_pdf)))

📊 TFT TRAITS TIER LIST - SET 15 (Sorted by Trait Impact)


📈 Tổng số traits phân tích: 26
📊 Hiển thị Top 50 traits có impact tốt nhất

🎨 Chú thích Place Change:
  🟢 Place Change ÂM (xanh) = Trait giúp placement TỐT HƠN
  🔴 Place Change DƯƠNG (đỏ) = Trait làm placement KÉM HƠN



#,Trait,Avg Place,Place Change,Win Rate %,Avg Tier,Count,Freq %,Levels
1,DragonFist,3.66,-0.81,65.0%,1.0,4822,2.85,T1
2,ElTigre,3.76,-0.71,63.2%,1.0,7667,4.53,T1
3,Luchador,3.82,-0.65,62.9%,1.1,4481,2.65,"T1, T2"
4,Captain,3.88,-0.59,59.7%,1.0,3805,2.25,T1
5,Rosemother,3.92,-0.55,59.0%,1.0,6503,3.84,T1
6,Destroyer,4.10,-0.37,58.8%,1.5,6865,4.06,"T1, T2, T3"
7,Bastion,4.19,-0.28,55.4%,1.5,11368,6.72,"T1, T3, T2"
8,Edgelord,4.20,-0.27,57.3%,1.1,4322,2.55,"T1, T2, T3"
9,Sniper,4.20,-0.27,56.2%,1.3,3349,1.98,"T1, T2, T3"
10,TheCrew,4.21,-0.26,54.4%,7.0,2119,1.25,T7


## Phân tích chi tiết: Trait và các Tier Levels

In [11]:
# Xem chi tiết một trait phổ biến
example_trait = "TFT15_Bastion"

print(f"=== PHÂN TÍCH CHI TIẾT: {example_trait} ===\n")

# Lấy thông tin trait từ trait_stats
trait_info = trait_stats.filter(col("trait_id") == example_trait).first()
if trait_info:
    print(f"📊 Thống kê tổng quan của {example_trait}:")
    print(f"   • Avg Place (overall): {trait_info['avg_place']}")
    print(f"   • Win Rate: {trait_info['win_rate']}%")
    print(f"   • Số lần xuất hiện: {trait_info['count']:,}")
    print(f"   • Avg Tier Level: {trait_info['avg_tier']}")

# Lấy Place Change
pc_info = trait_place_change.filter(col("trait_id") == example_trait).first()
if pc_info:
    print(f"\n💡 Place Change: {pc_info['place_change']}")
    print(f"   ➜ Giá trị {'ÂM' if pc_info['place_change'] < 0 else 'DƯƠNG'} = Trait này {'GIÚP' if pc_info['place_change'] < 0 else 'KHÔNG GIÚP'} tốt hơn!")

# Xem hiệu quả theo tier level
print(f"\n🎯 Hiệu quả theo từng Tier Level của {example_trait}:")
trait_tier_stats.filter(col("trait_id") == example_trait) \
    .orderBy("tier_current") \
    .show(10, truncate=False)

=== PHÂN TÍCH CHI TIẾT: TFT15_Bastion ===

📊 Thống kê tổng quan của TFT15_Bastion:
   • Avg Place (overall): 4.19
   • Win Rate: 55.4%
   • Số lần xuất hiện: 11,368
   • Avg Tier Level: 1.5
📊 Thống kê tổng quan của TFT15_Bastion:
   • Avg Place (overall): 4.19
   • Win Rate: 55.4%
   • Số lần xuất hiện: 11,368
   • Avg Tier Level: 1.5



💡 Place Change: -0.28
   ➜ Giá trị ÂM = Trait này GIÚP tốt hơn!

🎯 Hiệu quả theo từng Tier Level của TFT15_Bastion:
+-------------+------------+-----+---------+--------+
|trait_id     |tier_current|count|avg_place|win_rate|
+-------------+------------+-----+---------+--------+
|TFT15_Bastion|1           |8516 |4.15     |55.9    |
|TFT15_Bastion|2           |364  |5.36     |32.7    |
|TFT15_Bastion|3           |2488 |4.16     |57.2    |
+-------------+------------+-----+---------+--------+

+-------------+------------+-----+---------+--------+
|trait_id     |tier_current|count|avg_place|win_rate|
+-------------+------------+-----+---------+--------+
|TFT15_Bastion|1           |8516 |4.15     |55.9    |
|TFT15_Bastion|2           |364  |5.36     |32.7    |
|TFT15_Bastion|3           |2488 |4.16     |57.2    |
+-------------+------------+-----+---------+--------+



## Thống kê theo Tier Classification

In [12]:
from pyspark.sql.functions import when, count as spark_count_agg

# Phân loại tier dựa trên avg_place
print("\n📊 THỐNG KÊ THEO TIER:")
tier_classification = popular_traits.withColumn(
    "Tier",
    when(col("avg_place") < 3.80, "S")
    .when((col("avg_place") >= 3.80) & (col("avg_place") < 4.00), "A")
    .when((col("avg_place") >= 4.00) & (col("avg_place") < 4.20), "B")
    .when((col("avg_place") >= 4.20) & (col("avg_place") < 4.40), "C")
    .otherwise("D")
).groupBy("Tier").agg(
    spark_count_agg("*").alias("Count"),
    spark_round(avg("win_rate"), 1).alias("Avg Win Rate"),
    spark_round(avg("avg_place"), 2).alias("Avg Place")
).orderBy("Tier")

tier_classification.show()


📊 THỐNG KÊ THEO TIER:


+----+-----+------------+---------+
|Tier|Count|Avg Win Rate|Avg Place|
+----+-----+------------+---------+
|   A|    3|        60.5|     3.87|
|   B|    2|        57.1|     4.15|
|   C|   13|        54.6|     4.27|
|   D|    6|        48.7|     4.53|
|   S|    2|        64.1|     3.71|
+----+-----+------------+---------+



In [13]:
# Dừng Spark Session
spark.stop()
print("✅ Đã hoàn thành phân tích Traits Tier List!")

✅ Đã hoàn thành phân tích Traits Tier List!
